### Dataset / Image / File

This cell documents the traffic scene object detection experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import json
import os
from pathlib import Path
from collections import defaultdict
import shutil  # 用于复制图像

### Parameters / Path

This cell documents the traffic scene object detection experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [3]:
# Note on Dataset, Path, Data. Refer to the archived source for the detailed original comment.
dataset_dir = "/kaggle/input/datasets/xiaoyuwan/didi-detection/dataset/dataset_release"
# Note on Output. Refer to the archived source for the detailed original comment.
output_root = "/kaggle/working/dataset"

# Note on Create. Refer to the archived source for the detailed original comment.
os.makedirs(os.path.join(output_root, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(output_root, "images", "val"), exist_ok=True)
os.makedirs(os.path.join(output_root, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(output_root, "labels", "val"), exist_ok=True)

print(f"输出根目录已准备: {output_root}")

输出根目录已准备: /kaggle/working/dataset


### Experiment Notes

This cell documents the traffic scene object detection experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [4]:
def convert_coco_to_yolo(coco_json_path, images_dir, output_root, stage):
    """
    将 COCO 格式的数据集转换为 YOLO 格式，并复制图像到标准目录结构。

    COCO 格式:
    - images: [{"id": int, "file_name": str, "height": int, "width": int}, ...]
    - annotations: [{"image_id": int, "category_id": int, "bbox": [x, y, w, h]边界框（左下角 x, y，宽度 w，高度 h）, ...}, ...]
    - categories: [{"id": int, "name": str}, ...]

    YOLO 格式 (每个图像对应一个 .txt 文件):
    - 每行一个目标: <class_id> <x_center> <y_center> <width> <height>
    - 所有坐标都是相对于图像尺寸的归一化值 (0-1)

    Args:
        coco_json_path: COCO 标注文件路径 (如 val.json)
        images_dir: 图像文件夹路径
        output_dir: YOLO 格式输出目录
    """
    # 目标目录
    labels_dir = Path(output_root) / "labels" / stage
    images_target_dir = Path(output_root) / "images" / stage
    labels_dir.mkdir(parents=True, exist_ok=True)
    images_target_dir.mkdir(parents=True, exist_ok=True)

    # Note on Load. Refer to the archived source for the detailed original comment.
    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)

    images = coco_data['images']
    annotations = coco_data['annotations']
    categories = coco_data['categories']

    # 类别映射
    category_to_class = {cat['id']: idx for idx, cat in enumerate(categories)}

    # Note on Save, File. Refer to the archived source for the detailed original comment.
    classes_path = Path(output_root) / "labels" / "classes.txt"
    if not classes_path.exists():
        class_names = [cat['name'] for cat in categories]
        with open(classes_path, 'w') as f:
            for name in class_names:
                f.write(f"{name}\n")

    # Note on Image. Refer to the archived source for the detailed original comment.
    annotations_by_image = defaultdict(list)
    for ann in annotations:
        annotations_by_image[ann['image_id']].append(ann)

    # Note on Image. Refer to the archived source for the detailed original comment.
    converted_count = 0
    for image_info in images:
        img_id = image_info['id']
        file_name = image_info['file_name']
        img_width = image_info['width']
        img_height = image_info['height']

        # Note on Image. Refer to the archived source for the detailed original comment.
        src_img_path = Path(images_dir) / file_name
        if src_img_path.exists():
            dst_img_path = images_target_dir / file_name
            shutil.copy2(src_img_path, dst_img_path)  # 复制并保留元数据
        else:
            print(f"警告: 图像文件不存在 {src_img_path}")

        # Note on File, Generate. Refer to the archived source for the detailed original comment.
        base_name = Path(file_name).stem
        yolo_label_path = labels_dir / f"{base_name}.txt"
        image_annotations = annotations_by_image.get(img_id, [])

        yolo_lines = []
        for ann in image_annotations:
            # ======== TODO: 完成标注转换 ======== 归一化中心坐标
            class_id = category_to_class[ann['category_id'] ]

            x, y, w, h = ann['bbox']
            x_yolo = (x + w / 2) / img_width
            y_yolo = (y + h / 2) / img_height
            w_yolo = w / img_width
            h_yolo = h / img_height
            # ==================================
            
            yolo_lines.append(f"{class_id} {x_yolo:.6f} {y_yolo:.6f} {w_yolo:.6f} {h_yolo:.6f}")

        # Note on File. Refer to the archived source for the detailed original comment.
        with open(yolo_label_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

        converted_count += 1

    print("转换完成!")
    print(f"  - 图像数量: {len(images)}")
    print(f"  - 标注数量: {len(annotations)}")
    print(f"  - 类别数量: {len(categories)}")
    print(f"  - 生成的 YOLO 标签文件: {converted_count}")
    print(f"  - 标签目录: {labels_dir}")
    print(f"  - 图像目录: {images_target_dir}")

### Dataset / Data

This cell documents the traffic scene object detection experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [5]:
import os

dataset_dir = "/kaggle/input/datasets/xiaoyuwan/didi-detection/dataset/dataset_release"
print("目录内容:", os.listdir(dataset_dir))

目录内容: ['train.json', 'val', 'test', 'train', 'val.json']


In [6]:
for stage in ("train", "val"):
    print(f"\n处理 {stage} 数据集...")
    convert_coco_to_yolo(
        coco_json_path=os.path.join(dataset_dir, f"{stage}.json"),
        images_dir=os.path.join(dataset_dir, stage),
        output_root=output_root,
        stage=stage
    )


处理 train 数据集...
转换完成!
  - 图像数量: 10000
  - 标注数量: 82410
  - 类别数量: 12
  - 生成的 YOLO 标签文件: 10000
  - 标签目录: /kaggle/working/dataset/labels/train
  - 图像目录: /kaggle/working/dataset/images/train

处理 val 数据集...
转换完成!
  - 图像数量: 1000
  - 标注数量: 8608
  - 类别数量: 12
  - 生成的 YOLO 标签文件: 1000
  - 标签目录: /kaggle/working/dataset/labels/val
  - 图像目录: /kaggle/working/dataset/images/val


### Results / Validation / File

This cell documents the traffic scene object detection experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [7]:
# Note on File. Refer to the archived source for the detailed original comment.
print("类别列表:")
with open(Path(output_root) / "labels" / "classes.txt", 'r') as f:
    print(f.read())

类别列表:
car
van
bus
truck
person
bicycle
motorcycle
open-tricycle
closed-tricycle
forklift
large-block
small-block



In [8]:
# Note on File, Generate. Refer to the archived source for the detailed original comment.
labels_dir = Path(output_root) / "labels" / "val"
label_files = list(labels_dir.glob("*.txt"))
print(f"生成的标注文件数量: {len(label_files)}")
print(f"前 10 个文件: {[f.name for f in label_files[:10]]}")

生成的标注文件数量: 1000
前 10 个文件: ['10545.txt', '10243.txt', '10861.txt', '10762.txt', '10476.txt', '10660.txt', '10827.txt', '10819.txt', '10774.txt', '10962.txt']


In [9]:
# Note on File. Refer to the archived source for the detailed original comment.
if label_files:
    sample_file = label_files[0]
    print(f"\n文件: {sample_file.name}")
    print("内容:")
    with open(sample_file, 'r') as f:
        print(f.read())


文件: 10545.txt
内容:
2 0.526391 0.519653 0.046281 0.090306
3 0.428152 0.496521 0.135383 0.291069
0 0.492086 0.536660 0.007078 0.013347
0 0.498164 0.531493 0.008984 0.017458


### Validation / Visualization / Plot

This cell documents the traffic scene object detection experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [10]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def visualize_yolo_label(image_path, label_path, class_names):
    """可视化 YOLO 标注"""
    # Note on Image, Load. Refer to the archived source for the detailed original comment.
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Note on Load. Refer to the archived source for the detailed original comment.
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    # Note on Plot. Refer to the archived source for the detailed original comment.
    for line in lines:
        parts = line.strip().split()
        if len(parts) == 5:
            class_id = int(parts[0])
            x_center, y_center, bw, bh = map(float, parts[1:])
            
            # 转换为像素坐标
            x1 = int((x_center - bw/2) * w)
            y1 = int((y_center - bh/2) * h)
            x2 = int((x_center + bw/2) * w)
            y2 = int((y_center + bh/2) * h)
            
            # Note on Plot. Refer to the archived source for the detailed original comment.
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Note on Plot. Refer to the archived source for the detailed original comment.
            label = class_names[class_id] if class_id < len(class_names) else str(class_id)
            cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 4)
    
    return img


# Note on Visualization, Image. Refer to the archived source for the detailed original comment.
if label_files:
    sample_label = label_files[0]
    image_name = sample_label.stem + ".jpg"
    # Note on Path. Refer to the archived source for the detailed original comment.
    stage = sample_label.parent.name  # 例如 "val"
    # Note on Image, Path. Refer to the archived source for the detailed original comment.
    image_path = Path(dataset_dir) / stage / image_name
    
    if image_path.exists():
        # Note on Load. Refer to the archived source for the detailed original comment.
        with open(Path(output_root) / "labels" / "classes.txt", "r") as f:
            class_names = [line.strip() for line in f.readlines()]
        
        img_with_boxes = visualize_yolo_label(image_path, sample_label, class_names)
        
        plt.figure(figsize=(12, 8))
        plt.imshow(img_with_boxes)
        plt.title(f"Image: {image_name}")
        plt.axis('off')
        plt.show()
    else:
        print(f"图像文件不存在: {image_path}")

<Figure size 1200x800 with 1 Axes>